# <center> Семинар 15. Объясняемые рекомендации на практике </center>

## Цель семинара

Сегодня мы не пытаемся построить самый сильный recommender. Вместо этого соберём небольшой, но реалистичный пайплайн на KION и посмотрим, **как объяснять рекомендации**, которые дают разные классы моделей.

Что будет внутри:

- подготовка implicit-feedback датасета KION;
- обучение простых моделей: **EASE**, **iALS** и **градиентный бустинг**;
- несколько типов объяснений:
  - `because you watched ...` через вклад прошлых айтемов;
  - похожесть в latent factors;
  - важность и локальный вклад признаков в бустинге;
  - human-readable шаблоны по жанру, стране, году, длительности.

Главная мысль семинара: объяснение почти всегда зависит от того, **какой сигнал использует модель**. У линейной item-item модели объяснение одно, у факторизационной модели другое, у feature-based ранкера третье.

## План

1. Загрузим KION: `interactions.csv`, `items.csv`, `users.csv`.
2. Сделаем leave-last-out split: train = история, valid/test = следующие просмотры.
3. Обучим EASE и посмотрим, какие прошлые айтемы дали вклад в рекомендацию.
4. Обучим iALS, если установлен `implicit`, и объясним рекомендацию через похожие просмотренные айтемы в latent space.
5. Обучим градиентный бустинг на user/item/context features и разберём рекомендации через признаки.


In [1]:
# Если работаете в Colab / чистом окружении, можно раскомментировать:
# !pip install -q pandas numpy scipy scikit-learn matplotlib tqdm implicit shap

import os

# Важно выставить до импорта numpy/scipy/implicit.
# На некоторых машинах implicit + OpenBLAS/OpenMP может ронять kernel из-за oversubscription потоков.
for var in [
    "OPENBLAS_NUM_THREADS",
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ.setdefault(var, "1")

import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

## 1. Загрузка KION

В разных семинарах файлы могут лежать рядом с ноутбуком, на уровень выше или в `/mnt/data`. Поэтому используем устойчивый загрузчик, как в `seminar12`.

In [2]:
def find_file(candidates: List[str]) -> Optional[str]:
    search_roots = [Path("."), Path(".."), Path("../.."), Path("/mnt/data")]
    for root in search_roots:
        for name in candidates:
            path = root / name
            if path.exists():
                return str(path)
    return None

interactions_path = find_file(["interactions.csv", "kion_interactions.csv"])
items_path = find_file(["items.csv", "kion_items.csv"])
users_path = find_file(["users.csv", "kion_users.csv"])

print("interactions_path =", interactions_path)
print("items_path        =", items_path)
print("users_path        =", users_path)

if interactions_path is None:
    raise FileNotFoundError(
        "Не найден interactions.csv. Положите KION files рядом с ноутбуком, в ../, ../../ или /mnt/data."
    )

interactions_raw = pd.read_csv(interactions_path)
items_raw = pd.read_csv(items_path) if items_path is not None else pd.DataFrame()
users_raw = pd.read_csv(users_path) if users_path is not None else pd.DataFrame()

print("Interactions:", interactions_raw.shape)
print("Items       :", items_raw.shape)
print("Users       :", users_raw.shape)

display(interactions_raw.head())
display(items_raw.head())
display(users_raw.head())

interactions_path = ../../interactions.csv
items_path        = ../../items.csv
users_path        = ../../users.csv
Interactions: (5476251, 5)
Items       : (15963, 14)
Users       : (840197, 5)


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


## 2. Нормализация колонок

Нам нужны три обязательные сущности: пользователь, айтем и время. Всё остальное используем как дополнительные признаки для объяснений.

In [3]:
def detect_column(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    cols = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in cols:
            return cols[candidate.lower()]
    if required:
        raise ValueError(f"Не удалось найти колонку из кандидатов: {candidates}")
    return None

USER_COL = detect_column(interactions_raw, ["user_id", "uid", "profile_id"])
ITEM_COL = detect_column(interactions_raw, ["item_id", "asset_id", "movie_id", "content_id"])
TIME_COL = detect_column(interactions_raw, ["last_watch_dt", "event_time", "timestamp", "datetime", "watch_dt"])
WATCH_COL = detect_column(
    interactions_raw,
    ["watch_duration", "watched_pct", "total_dur", "watchtime", "duration", "watched_time"],
    required=False,
)

interactions = interactions_raw[[c for c in [USER_COL, ITEM_COL, TIME_COL, WATCH_COL] if c is not None]].copy()
interactions = interactions.rename(columns={
    USER_COL: "user_id",
    ITEM_COL: "item_id",
    TIME_COL: "event_time",
    **({WATCH_COL: "watch_value"} if WATCH_COL is not None else {}),
})
interactions["event_time"] = pd.to_datetime(interactions["event_time"])
interactions["user_id"] = interactions["user_id"].astype(str)
interactions["item_id"] = interactions["item_id"].astype(str)
if "watch_value" in interactions.columns:
    interactions["watch_value"] = pd.to_numeric(interactions["watch_value"], errors="coerce")
else:
    interactions["watch_value"] = 1.0

interactions = interactions.sort_values(["user_id", "event_time"]).drop_duplicates(["user_id", "item_id"], keep="last")

print("USER_COL =", USER_COL)
print("ITEM_COL =", ITEM_COL)
print("TIME_COL =", TIME_COL)
print("WATCH_COL =", WATCH_COL)
display(interactions.head())

USER_COL = user_id
ITEM_COL = item_id
TIME_COL = last_watch_dt
WATCH_COL = watched_pct


,user_id,item_id,event_time,watch_value
3590116,0,12192,2021-07-16,0.0
620,0,7102,2021-07-19,3.0
67070,0,14359,2021-07-19,2.0
90113,0,15297,2021-07-19,0.0
3103040,0,9728,2021-07-19,0.0


In [4]:
def normalize_items(items: pd.DataFrame) -> pd.DataFrame:
    if items.empty:
        return pd.DataFrame({"item_id": interactions["item_id"].unique()})

    item_col = detect_column(items, ["item_id", "asset_id", "movie_id", "content_id"])
    out = items.copy().rename(columns={item_col: "item_id"})
    out["item_id"] = out["item_id"].astype(str)

    rename_candidates = {
        "title": ["title", "name", "movie_name", "content_name"],
        "genres": ["genres", "genre"],
        "countries": ["countries", "country"],
        "release_year": ["release_year", "year"],
        "duration": ["duration", "runtime", "item_duration"],
    }
    for target, candidates in rename_candidates.items():
        col = detect_column(out, candidates, required=False)
        if col is not None and col != target:
            out = out.rename(columns={col: target})

    if "title" not in out.columns:
        out["title"] = "item " + out["item_id"].astype(str)
    for col in ["genres", "countries"]:
        if col not in out.columns:
            out[col] = "unknown"
    for col in ["release_year", "duration"]:
        if col not in out.columns:
            out[col] = np.nan

    return out

items = normalize_items(items_raw)
item_title = items.set_index("item_id")["title"].to_dict()

def title(item_id: str) -> str:
    return str(item_title.get(str(item_id), f"item {item_id}"))

display(items[["item_id", "title", "genres", "countries", "release_year", "duration"]].head())

,item_id,title,genres,countries,release_year,duration
0,10711,Поговори с ней,"драмы, зарубежные, детективы, мелодрамы",Испания,2002.0,NaN
1,2508,Голые перцы,"зарубежные, приключения, комедии",США,2014.0,NaN
2,10716,Тактическая сила,"криминал, зарубежные, триллеры, боевики, комедии",Канада,2011.0,NaN
3,7868,45 лет,"драмы, зарубежные, мелодрамы",Великобритания,2015.0,NaN
4,16268,Все решает мгновение,"драмы, спорт, советские, мелодрамы",СССР,1978.0,NaN


## 3. Семинарский sampling и leave-last-out split

Чтобы ноутбук запускался на обычном ноутбуке, ограничим число пользователей и число айтемов. Для настоящего эксперимента эти лимиты можно увеличить.

In [5]:
MAX_USERS = 12_000
MAX_ITEMS = 3_000
MIN_USER_EVENTS = 5

user_sizes = interactions.groupby("user_id").size()
good_users = user_sizes[user_sizes >= MIN_USER_EVENTS].index.to_numpy()
if len(good_users) > MAX_USERS:
    good_users = rng.choice(good_users, size=MAX_USERS, replace=False)

data = interactions[interactions["user_id"].isin(good_users)].copy()

top_items = data["item_id"].value_counts().head(MAX_ITEMS).index
data = data[data["item_id"].isin(top_items)].copy()

# После фильтрации по top items снова удалим слишком короткие истории.
user_sizes = data.groupby("user_id").size()
good_users = user_sizes[user_sizes >= MIN_USER_EVENTS].index
data = data[data["user_id"].isin(good_users)].sort_values(["user_id", "event_time"]).copy()

print("Interactions:", len(data))
print("Users       :", data["user_id"].nunique())
print("Items       :", data["item_id"].nunique())
print("Avg seq len :", round(data.groupby("user_id").size().mean(), 2))

Interactions: 161506
Users       : 11648
Items       : 3000
Avg seq len : 13.87


In [6]:
def leave_last_out(frame: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    parts_train, parts_valid, parts_test = [], [], []
    for _, g in frame.groupby("user_id", sort=False):
        g = g.sort_values("event_time")
        parts_train.append(g.iloc[:-2])
        parts_valid.append(g.iloc[-2:-1])
        parts_test.append(g.iloc[-1:])
    return pd.concat(parts_train), pd.concat(parts_valid), pd.concat(parts_test)

train_df, valid_df, test_df = leave_last_out(data)

all_user_ids = sorted(data["user_id"].unique())
all_item_ids = sorted(data["item_id"].unique())
user2idx = {u: i for i, u in enumerate(all_user_ids)}
item2idx = {it: i for i, it in enumerate(all_item_ids)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: it for it, i in item2idx.items()}

print(train_df.shape, valid_df.shape, test_df.shape)

(138210, 4) (11648, 4) (11648, 4)


In [7]:
def build_user_item_matrix(frame: pd.DataFrame, weight_col: str = "watch_value") -> sparse.csr_matrix:
    rows = frame["user_id"].map(user2idx).to_numpy()
    cols = frame["item_id"].map(item2idx).to_numpy()
    if weight_col in frame.columns and frame[weight_col].notna().any():
        values = frame[weight_col].fillna(1.0).to_numpy(dtype=np.float32)
        q95 = np.quantile(values, 0.95) if len(values) else 1.0
        values = np.clip(values / max(q95, 1e-6), 0.0, 1.0)
    else:
        values = np.ones(len(frame), dtype=np.float32)
    return sparse.csr_matrix((values, (rows, cols)), shape=(len(user2idx), len(item2idx)))

X_train = build_user_item_matrix(train_df)
X_train_bin = X_train.copy()
X_train_bin.data = np.ones_like(X_train_bin.data)

user_seen = {
    user2idx[u]: set(g["item_id"].map(item2idx))
    for u, g in train_df.groupby("user_id")
}
valid_target = valid_df.set_index("user_id")["item_id"].map(item2idx).to_dict()
test_target = test_df.set_index("user_id")["item_id"].map(item2idx).to_dict()

X_train.shape, X_train.nnz

((11648, 3000), 138210)

## 4. Метрики и утилиты для рекомендаций

Для семинара достаточно `Recall@K`: попал ли единственный следующий айтем пользователя в top-K.

In [8]:
def mask_seen(scores: np.ndarray, user_idx: int) -> np.ndarray:
    scores = scores.copy()
    seen = user_seen.get(user_idx, set())
    if seen:
        scores[list(seen)] = -np.inf
    return scores


def top_k_from_scores(scores: np.ndarray, k: int = 10) -> List[int]:
    if k >= len(scores):
        return list(np.argsort(-scores))
    candidates = np.argpartition(-scores, kth=k)[:k]
    return candidates[np.argsort(-scores[candidates])].tolist()


def recall_at_k(score_fn, targets: Dict[str, int], k: int = 10, max_users: Optional[int] = 2000) -> float:
    users = list(targets.keys())
    if max_users is not None and len(users) > max_users:
        users = list(rng.choice(users, size=max_users, replace=False))
    hits = []
    for user_id in tqdm(users, desc=f"Recall@{k}"):
        u = user2idx[user_id]
        scores = mask_seen(score_fn(u), u)
        recs = top_k_from_scores(scores, k)
        hits.append(int(targets[user_id] in recs))
    return float(np.mean(hits))


def show_recs(user_idx: int, recs: List[int], scores: Optional[np.ndarray] = None) -> pd.DataFrame:
    rows = []
    for rank, item_idx in enumerate(recs, start=1):
        item_id = idx2item[item_idx]
        rows.append({
            "rank": rank,
            "item_id": item_id,
            "title": title(item_id),
            "score": None if scores is None else float(scores[item_idx]),
        })
    return pd.DataFrame(rows)

# Часть A. EASE: объяснение через вклад прошлых айтемов

EASE учит матрицу item-item весов `B`. Скор пользователя для кандидата `j`:

$$ score(u, j) = \sum_{i \in history(u)} X_{ui} B_{ij} $$

Это удобно объяснять: можно показать, какие просмотренные айтемы дали самый большой вклад в рекомендацию.

In [9]:
class EASE:
    def __init__(self, reg: float = 500.0):
        self.reg = reg
        self.B = None

    def fit(self, X: sparse.csr_matrix):
        G = (X.T @ X).toarray().astype(np.float64)
        diag = np.diag_indices(G.shape[0])
        G[diag] += self.reg
        P = np.linalg.inv(G)
        B = -P / np.diag(P)
        B[diag] = 0.0
        self.B = B.astype(np.float32)
        return self

    def user_scores(self, user_idx: int) -> np.ndarray:
        return X_train_bin[user_idx].toarray().ravel() @ self.B

    def explain(self, user_idx: int, item_idx: int, top_n: int = 5) -> pd.DataFrame:
        history = X_train_bin[user_idx].indices
        contrib = self.B[history, item_idx]
        order = np.argsort(-contrib)[:top_n]
        rows = []
        for pos in order:
            hist_idx = int(history[pos])
            hist_item = idx2item[hist_idx]
            rows.append({
                "watched_item_id": hist_item,
                "watched_title": title(hist_item),
                "contribution": float(contrib[pos]),
            })
        return pd.DataFrame(rows)

In [10]:
ease = EASE(reg=700.0).fit(X_train_bin)
print("EASE valid Recall@10:", recall_at_k(ease.user_scores, valid_target, k=10, max_users=1500))

Recall@10:   0%|          | 0/1500 [00:00<?, ?it/s]

EASE valid Recall@10: 0.254


In [15]:
example_user_id = valid_df["user_id"].iloc[1]
example_user_idx = user2idx[example_user_id]

ease_scores = mask_seen(ease.user_scores(example_user_idx), example_user_idx)
ease_recs = top_k_from_scores(ease_scores, k=10)

print("User:", example_user_id)
display(show_recs(example_user_idx, ease_recs, ease_scores))

chosen_item_idx = ease_recs[0]
print("Почему рекомендовали:", title(idx2item[chosen_item_idx]))
display(ease.explain(example_user_idx, chosen_item_idx, top_n=7))

User: 1000279


,rank,item_id,title,score
0,1,15297,Клиника счастья,0.518591
1,2,3734,Прабабушка легкого поведения,0.315620
2,3,7571,100% волк,0.243608
3,4,2657,Подслушано,0.201345
4,5,4880,Афера,0.198872
5,6,142,Маша,0.196949
6,7,10761,Моана,0.169174
7,8,13243,Головоломка,0.160658
8,9,9996,Немцы,0.159170
9,10,16270,Тайна Коко,0.159038


Почему рекомендовали: Клиника счастья


,watched_item_id,watched_title,contribution
0,4151,Секреты семейной жизни,0.171355
1,10440,Хрустальный,0.144569
2,13865,Девятаев,0.047751
3,9728,Гнев человеческий,0.045707
4,8636,Белый снег,0.033897
5,6809,Дуров,0.031273
6,11863,Девятаев - сериал,0.030356


In [16]:
def textual_ease_explanation(user_idx: int, item_idx: int, n: int = 3) -> str:
    exp = ease.explain(user_idx, item_idx, top_n=n)
    watched = "; ".join(exp["watched_title"].astype(str).tolist())
    return f"Рекомендуем '{title(idx2item[item_idx])}', потому что в истории есть похожие просмотры: {watched}."

for item_idx in ease_recs[:3]:
    print(textual_ease_explanation(example_user_idx, item_idx))

Рекомендуем 'Клиника счастья', потому что в истории есть похожие просмотры: Секреты семейной жизни; Хрустальный; Девятаев.
Рекомендуем 'Прабабушка легкого поведения', потому что в истории есть похожие просмотры: Гнев человеческий; Девятаев; Секреты семейной жизни.
Рекомендуем '100% волк', потому что в истории есть похожие просмотры: Ральф против Интернета; Зверополис; Белый снег.


# Часть B. iALS: объяснение через похожие прошлые айтемы в latent space

Implicit ALS учит эмбеддинги пользователей и айтемов. Прямой item-level вклад здесь не такой прозрачный, как в EASE, но можно сделать практичное объяснение:

1. берём рекомендованный айтем;
2. ищем в истории пользователя айтемы с максимальным dot product / cosine similarity с ним;
3. показываем их как evidence.

Это не строгое причинное объяснение, но часто именно так делают production-friendly explanations для embedding models.

In [17]:
try:
    from implicit.als import AlternatingLeastSquares
    HAS_IMPLICIT = True
except Exception as e:
    HAS_IMPLICIT = False
    print("implicit не установлен. Блок iALS можно пропустить или установить: pip install implicit")
    print(type(e).__name__, e)

In [18]:
if HAS_IMPLICIT:
    als = AlternatingLeastSquares(
        factors=48,
        regularization=0.08,
        iterations=12,
        num_threads=1,
        random_state=RANDOM_STATE,
    )
    # implicit ALS ожидает user-item matrix: строки = пользователи, колонки = айтемы.
    als.fit((X_train_bin * 40.0).astype(np.float32))

    def als_scores(user_idx: int) -> np.ndarray:
        user_vec = als.user_factors[user_idx]
        return als.item_factors @ user_vec

    print("ALS valid Recall@10:", recall_at_k(als_scores, valid_target, k=10, max_users=1500))
else:
    als = None

  0%|          | 0/12 [00:00<?, ?it/s]

Recall@10:   0%|          | 0/1500 [00:00<?, ?it/s]

ALS valid Recall@10: 0.17733333333333334


In [19]:
def explain_als_by_history(user_idx: int, item_idx: int, top_n: int = 5) -> pd.DataFrame:
    if als is None:
        return pd.DataFrame()

    # Самый faithful вариант для implicit ALS: встроенный explain раскладывает score
    # рекомендованного item на вклады item-ов из истории пользователя.
    try:
        total_score, contributions, _ = als.explain(user_idx, X_train_bin, item_idx, N=top_n)
        rows = []
        for hist_idx, contribution in contributions:
            hist_item = idx2item[int(hist_idx)]
            rows.append({
                "watched_item_id": hist_item,
                "watched_title": title(hist_item),
                "als_contribution": float(contribution),
                "total_score": float(total_score),
                "explanation_type": "implicit.als.explain",
            })
        return pd.DataFrame(rows)
    except Exception as e:
        print("als.explain не сработал, используем fallback через cosine similarity:", type(e).__name__, e)

    # Fallback: это не точное разложение ALS-score, а понятная эвристика
    # 'рекомендованный айтем близок к этим просмотренным айтемам в latent space'.
    history = list(user_seen.get(user_idx, []))
    target_vec = als.item_factors[item_idx]
    hist_vecs = als.item_factors[history]
    denom = np.linalg.norm(hist_vecs, axis=1) * max(np.linalg.norm(target_vec), 1e-12)
    sims = (hist_vecs @ target_vec) / np.maximum(denom, 1e-12)
    order = np.argsort(-sims)[:top_n]
    rows = []
    for pos in order:
        hist_idx = int(history[pos])
        hist_item = idx2item[hist_idx]
        rows.append({
            "watched_item_id": hist_item,
            "watched_title": title(hist_item),
            "latent_cosine_similarity": float(sims[pos]),
            "explanation_type": "latent-neighbor fallback",
        })
    return pd.DataFrame(rows)

if HAS_IMPLICIT:
    als_scores_u = mask_seen(als_scores(example_user_idx), example_user_idx)
    als_recs = top_k_from_scores(als_scores_u, k=10)
    display(show_recs(example_user_idx, als_recs, als_scores_u))
    print("Почему рекомендовали:", title(idx2item[als_recs[0]]))
    display(explain_als_by_history(example_user_idx, als_recs[0], top_n=7))

,rank,item_id,title,score
0,1,5411,Монстры на каникулах 3: Море зовёт,1.218010
1,2,13915,Вперёд,1.191715
2,3,7829,Поступь хаоса,1.164173
3,4,10761,Моана,1.135943
4,5,16270,Тайна Коко,1.032081
5,6,6646,Симпсоны в кино,0.974737
6,7,7571,100% волк,0.971283
7,8,4436,Ford против Ferrari,0.962048
8,9,11749,Суперсемейка 2,0.916966
9,10,10214,Мы – монстры,0.884311


Почему рекомендовали: Монстры на каникулах 3: Море зовёт


,watched_item_id,watched_title,als_contribution,total_score,explanation_type
0,3182,Ральф против Интернета,0.026061,0.170944,implicit.als.explain
1,16166,Зверополис,0.025846,0.170944,implicit.als.explain
2,14942,История игрушек: Большой побег,0.017353,0.170944,implicit.als.explain
3,4718,Вверх,0.016615,0.170944,implicit.als.explain
4,9164,ВАЛЛ-И,0.016339,0.170944,implicit.als.explain
5,15266,Корпорация монстров,0.015342,0.170944,implicit.als.explain
6,334,Храбрая сердцем,0.013417,0.170944,implicit.als.explain


# Часть C. Градиентный бустинг: объяснение через признаки

Теперь построим candidate reranker. Он получает пару `(user, item)` и признаки:

- насколько айтем популярен;
- насколько пользователь активен;
- пересекаются ли жанры кандидата с жанрами истории пользователя;
- год, длительность, свежесть;
- score от EASE как сильный collaborative feature.

Такой ранкер хорошо объясняется через feature contributions: можно показать, какие признаки подняли кандидата.

In [20]:
def split_tokens(x) -> List[str]:
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    text = str(x)
    for sep in [",", "|", "/"]:
        if sep in text:
            return [p.strip() for p in text.split(sep) if p.strip()]
    return [text.strip()] if text.strip() else []

items_feat = items[items["item_id"].isin(all_item_ids)].copy()
items_feat["genres_list"] = items_feat["genres"].apply(split_tokens)
items_feat["countries_list"] = items_feat["countries"].apply(split_tokens)
items_feat["release_year"] = pd.to_numeric(items_feat["release_year"], errors="coerce")
items_feat["duration"] = pd.to_numeric(items_feat["duration"], errors="coerce")

item_pop = train_df["item_id"].value_counts().rename("item_popularity")
item_watch_mean = train_df.groupby("item_id")["watch_value"].mean().rename("item_watch_mean")
items_feat = items_feat.merge(item_pop, left_on="item_id", right_index=True, how="left")
items_feat = items_feat.merge(item_watch_mean, left_on="item_id", right_index=True, how="left")
items_feat["item_popularity"] = items_feat["item_popularity"].fillna(0)
items_feat["item_watch_mean"] = items_feat["item_watch_mean"].fillna(items_feat["item_watch_mean"].median())

item_features = items_feat.set_index("item_id")

user_activity = train_df.groupby("user_id").size().rename("user_activity")
user_mean_watch = train_df.groupby("user_id")["watch_value"].mean().rename("user_mean_watch")
user_last_time = train_df.groupby("user_id")["event_time"].max().rename("user_last_time")

user_genres = {}
for user_id, g in train_df.merge(items_feat[["item_id", "genres_list"]], on="item_id", how="left").groupby("user_id"):
    counts = defaultdict(int)
    for genres in g["genres_list"]:
        for genre in genres:
            counts[genre] += 1
    user_genres[user_id] = counts

In [21]:
def genre_overlap(user_id: str, item_id: str) -> float:
    genres = item_features.loc[item_id, "genres_list"] if item_id in item_features.index else []
    if not genres:
        return 0.0
    counts = user_genres.get(user_id, {})
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return float(sum(counts.get(g, 0) for g in genres) / total)


EASE_SCORE_CACHE = {}

def cached_ease_scores(user_id: str) -> np.ndarray:
    if user_id not in EASE_SCORE_CACHE:
        EASE_SCORE_CACHE[user_id] = ease.user_scores(user2idx[user_id])
    return EASE_SCORE_CACHE[user_id]


def make_features(pairs: pd.DataFrame, add_ease_score: bool = True) -> pd.DataFrame:
    rows = []
    for r in pairs.itertuples(index=False):
        user_id, item_id = str(r.user_id), str(r.item_id)
        item_row = item_features.loc[item_id] if item_id in item_features.index else None
        iidx = item2idx[item_id]
        ease_score = float(cached_ease_scores(user_id)[iidx]) if add_ease_score else 0.0
        rows.append({
            "user_activity": float(user_activity.get(user_id, 0)),
            "user_mean_watch": float(user_mean_watch.get(user_id, 0)),
            "item_popularity": float(0 if item_row is None else item_row["item_popularity"]),
            "item_watch_mean": float(0 if item_row is None else item_row["item_watch_mean"]),
            "release_year": float(np.nan if item_row is None else item_row["release_year"]),
            "duration": float(np.nan if item_row is None else item_row["duration"]),
            "genre_overlap": genre_overlap(user_id, item_id),
            "ease_score": ease_score,
        })
    return pd.DataFrame(rows)

FEATURES = [
    "user_activity", "user_mean_watch", "item_popularity", "item_watch_mean",
    "release_year", "duration", "genre_overlap", "ease_score",
]

In [22]:
train_items_by_user = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

def sample_training_pairs(positive_df: pd.DataFrame, n_neg_per_pos: int = 4) -> pd.DataFrame:
    all_items_arr = np.array(all_item_ids)
    rows = []
    for r in tqdm(positive_df.itertuples(index=False), total=len(positive_df), desc="Sampling pairs"):
        user_id = str(r.user_id)
        pos_item = str(r.item_id)
        seen_items = train_items_by_user.get(user_id, set())
        rows.append({"user_id": user_id, "item_id": pos_item, "target": 1})
        negs = []
        while len(negs) < n_neg_per_pos:
            candidate = str(rng.choice(all_items_arr))
            if candidate not in seen_items and candidate != pos_item:
                negs.append(candidate)
        for neg_item in negs:
            rows.append({"user_id": user_id, "item_id": neg_item, "target": 0})
    return pd.DataFrame(rows)

# Для скорости учим ранкер на valid targets как positives: модель видит историю train -> следующий айтем.
gb_pairs = sample_training_pairs(valid_df, n_neg_per_pos=4)
X_gb = make_features(gb_pairs)
y_gb = gb_pairs["target"].astype(int).to_numpy()

ranker = HistGradientBoostingClassifier(
    max_iter=180,
    learning_rate=0.06,
    max_leaf_nodes=31,
    random_state=RANDOM_STATE,
)
ranker.fit(X_gb[FEATURES], y_gb)

pred = ranker.predict_proba(X_gb[FEATURES])[:, 1]
print("Train pairwise AUC:", roc_auc_score(y_gb, pred))

Sampling pairs:   0%|          | 0/11648 [00:00<?, ?it/s]

Train pairwise AUC: 0.899088012525791


In [24]:
popular_item_indices = [item2idx[x] for x in train_df["item_id"].value_counts().head(300).index if x in item2idx]

def gb_candidate_shortlist(user_idx: int, n_ease: int = 400) -> List[int]:
    # В production ранкер почти всегда работает не по всему каталогу, а по shortlist от retrieval-модели.
    ease_short = top_k_from_scores(mask_seen(ease.user_scores(user_idx), user_idx), k=min(n_ease, len(all_item_ids)))
    candidates = list(dict.fromkeys(ease_short + popular_item_indices))
    seen = user_seen.get(user_idx, set())
    return [i for i in candidates if i not in seen]


def gb_scores(user_idx: int, candidate_items: Optional[List[int]] = None) -> np.ndarray:
    user_id = idx2user[user_idx]
    if candidate_items is None:
        candidate_items = gb_candidate_shortlist(user_idx)
    pairs = pd.DataFrame({
        "user_id": [user_id] * len(candidate_items),
        "item_id": [idx2item[i] for i in candidate_items],
    })
    feats = make_features(pairs)
    scores = ranker.predict_proba(feats[FEATURES])[:, 1]
    out = np.full(len(all_item_ids), -np.inf, dtype=np.float32)
    out[candidate_items] = scores
    return out

print("GB test Recall@10:", recall_at_k(gb_scores, test_target, k=10, max_users=800))

Recall@10:   0%|          | 0/800 [00:00<?, ?it/s]

GB test Recall@10: 0.22625


## 5. Локальные объяснения бустинга

Для локального объяснения есть несколько уровней сложности:

- простой уровень: показать значения признаков у кандидата;
- средний уровень: permutation importance / global importance;
- продвинутый уровень: SHAP values.

В семинаре используем SHAP, если библиотека установлена. Если нет, fallback — отклонение признака от среднего и глобальная permutation importance.

In [25]:
try:
    import shap
    HAS_SHAP = True
except Exception as e:
    HAS_SHAP = False
    print("shap не установлен. Будем использовать fallback-объяснение.")
    print(type(e).__name__, e)

In [26]:
def explain_gb_candidate(user_idx: int, item_idx: int) -> pd.DataFrame:
    user_id = idx2user[user_idx]
    item_id = idx2item[item_idx]
    pair = pd.DataFrame({"user_id": [user_id], "item_id": [item_id]})
    x = make_features(pair)[FEATURES]

    if HAS_SHAP:
        try:
            explainer = shap.Explainer(ranker)
            values = explainer(x)
            contrib = values.values[0]
            base = values.base_values[0]
            out = pd.DataFrame({
                "feature": FEATURES,
                "value": x.iloc[0].values,
                "shap_contribution": contrib,
            }).sort_values("shap_contribution", ascending=False)
            out.attrs["base_value"] = base
            return out
        except Exception as e:
            print("SHAP не смог объяснить эту модель, используем fallback:", type(e).__name__, e)

    # Fallback: не SHAP, но полезный диагностический срез.
    means = X_gb[FEATURES].mean()
    stds = X_gb[FEATURES].std().replace(0, 1)
    z = ((x.iloc[0] - means) / stds).rename("z_score")
    return pd.DataFrame({
        "feature": FEATURES,
        "value": x.iloc[0].values,
        "z_score_vs_training_pairs": z.values,
    }).sort_values("z_score_vs_training_pairs", ascending=False)


gb_scores_u = mask_seen(gb_scores(example_user_idx), example_user_idx)
gb_recs = top_k_from_scores(gb_scores_u, k=10)
display(show_recs(example_user_idx, gb_recs, gb_scores_u))

chosen_gb_item = gb_recs[0]
print("Почему бустинг поднял:", title(idx2item[chosen_gb_item]))
display(explain_gb_candidate(example_user_idx, chosen_gb_item))

,rank,item_id,title,score
0,1,15297,Клиника счастья,0.959167
1,2,3734,Прабабушка легкого поведения,0.952751
2,3,5411,Монстры на каникулах 3: Море зовёт,0.920776
3,4,7571,100% волк,0.918790
4,5,13159,Рататуй,0.918150
5,6,16270,Тайна Коко,0.913923
6,7,142,Маша,0.906020
7,8,11749,Суперсемейка 2,0.899874
8,9,13243,Головоломка,0.899125
9,10,7582,Холодное сердце II,0.896359


Почему бустинг поднял: Клиника счастья


,feature,value,shap_contribution
7,ease_score,0.518591,3.770226
2,item_popularity,2715.000000,1.999455
4,release_year,2021.000000,0.130423
1,user_mean_watch,75.280000,0.083610
3,item_watch_mean,60.320442,0.077456
5,duration,NaN,0.000000
6,genre_overlap,0.133333,-0.138260
0,user_activity,25.000000,-0.823076


In [27]:
def describe_item_metadata(item_id: str) -> str:
    if item_id not in item_features.index:
        return "нет metadata"
    row = item_features.loc[item_id]
    genres = ", ".join(row["genres_list"][:3]) if isinstance(row["genres_list"], list) else str(row.get("genres", ""))
    countries = ", ".join(row["countries_list"][:2]) if isinstance(row["countries_list"], list) else str(row.get("countries", ""))
    year = "" if pd.isna(row["release_year"]) else str(int(row["release_year"]))
    return f"жанры: {genres}; страны: {countries}; год: {year}"


def textual_gb_explanation(user_idx: int, item_idx: int) -> str:
    user_id = idx2user[user_idx]
    item_id = idx2item[item_idx]
    features = make_features(pd.DataFrame({"user_id": [user_id], "item_id": [item_id]})).iloc[0]
    reasons = []
    if features["genre_overlap"] > 0:
        reasons.append(f"жанры похожи на историю пользователя (overlap={features['genre_overlap']:.2f})")
    if features["item_popularity"] >= X_gb["item_popularity"].quantile(0.75):
        reasons.append("айтем часто смотрели другие пользователи")
    if features["ease_score"] >= X_gb["ease_score"].quantile(0.75):
        reasons.append("collaborative-модель тоже дала высокий score")
    if not reasons:
        reasons.append("у айтема хороший общий feature-score для этого пользователя")
    return f"Рекомендуем '{title(item_id)}' ({describe_item_metadata(item_id)}), потому что " + "; ".join(reasons) + "."

for item_idx in gb_recs[:5]:
    print(textual_gb_explanation(example_user_idx, item_idx))

Рекомендуем 'Клиника счастья' (жанры: драмы, мелодрамы; страны: Россия; год: 2021), потому что жанры похожи на историю пользователя (overlap=0.13); айтем часто смотрели другие пользователи; collaborative-модель тоже дала высокий score.
Рекомендуем 'Прабабушка легкого поведения' (жанры: комедии; страны: Россия; год: 2021), потому что жанры похожи на историю пользователя (overlap=0.16); айтем часто смотрели другие пользователи; collaborative-модель тоже дала высокий score.
Рекомендуем 'Монстры на каникулах 3: Море зовёт' (жанры: мультфильм, фэнтези, приключения; страны: США; год: 2018), потому что жанры похожи на историю пользователя (overlap=0.59); айтем часто смотрели другие пользователи; collaborative-модель тоже дала высокий score.
Рекомендуем '100% волк' (жанры: мультфильм, приключения, семейное; страны: Австралия, Бельгия; год: 2020), потому что жанры похожи на историю пользователя (overlap=0.61); айтем часто смотрели другие пользователи; collaborative-модель тоже дала высокий scor

# Часть D. Сравнение объяснений

Для одного и того же пользователя разные модели могут рекомендовать разные айтемы и объяснять их разными языками.

Это нормально: объяснение должно быть согласовано с механизмом модели.

In [28]:
comparison_rows = []
for model_name, recs, scores in [
    ("EASE", ease_recs, ease_scores),
    ("GB", gb_recs, gb_scores_u),
]:
    for rank, item_idx in enumerate(recs[:5], start=1):
        comparison_rows.append({
            "model": model_name,
            "rank": rank,
            "title": title(idx2item[item_idx]),
            "score": float(scores[item_idx]),
            "metadata": describe_item_metadata(idx2item[item_idx]),
        })

if HAS_IMPLICIT:
    for rank, item_idx in enumerate(als_recs[:5], start=1):
        comparison_rows.append({
            "model": "ALS",
            "rank": rank,
            "title": title(idx2item[item_idx]),
            "score": float(als_scores_u[item_idx]),
            "metadata": describe_item_metadata(idx2item[item_idx]),
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

,model,rank,title,score,metadata
0,EASE,1,Клиника счастья,0.518591,"жанры: драмы, мелодрамы; страны: Россия; год: ..."
1,EASE,2,Прабабушка легкого поведения,0.315620,жанры: комедии; страны: Россия; год: 2021
2,EASE,3,100% волк,0.243608,"жанры: мультфильм, приключения, семейное; стра..."
3,EASE,4,Подслушано,0.201345,"жанры: драмы, триллеры; страны: Россия; год: 2021"
4,EASE,5,Афера,0.198872,жанры: комедии; страны: Россия; год: 2021
5,GB,1,Клиника счастья,0.959167,"жанры: драмы, мелодрамы; страны: Россия; год: ..."
6,GB,2,Прабабушка легкого поведения,0.952751,жанры: комедии; страны: Россия; год: 2021
7,GB,3,Монстры на каникулах 3: Море зовёт,0.920776,"жанры: мультфильм, фэнтези, приключения; стран..."
8,GB,4,100% волк,0.918790,"жанры: мультфильм, приключения, семейное; стра..."
9,GB,5,Рататуй,0.918150,"жанры: мультфильм, приключения, драмы; страны:..."


In [29]:
def plot_user_history(user_idx: int, n: int = 15):
    user_id = idx2user[user_idx]
    hist = train_df[train_df["user_id"] == user_id].sort_values("event_time").tail(n).copy()
    hist["title"] = hist["item_id"].map(title)
    display(hist[["event_time", "item_id", "title", "watch_value"]])

print("Последние просмотры пользователя")
plot_user_history(example_user_idx)

Последние просмотры пользователя


,event_time,item_id,title,watch_value
1952268,2021-08-01,9728,Гнев человеческий,100.0
5202554,2021-08-02,11863,Девятаев - сериал,0.0
5430657,2021-08-03,13865,Девятаев,100.0
3454540,2021-08-05,15603,Красавица и чудовище,100.0
872670,2021-08-05,8636,Белый снег,100.0
4885069,2021-08-07,13185,Хороший динозавр,100.0
5228753,2021-08-08,15266,Корпорация монстров,100.0
367654,2021-08-14,2802,Starперцы,96.0
370346,2021-08-14,7291,Концерт Jony,100.0
4331078,2021-08-14,10323,Университет монстров,100.0
